# データ探索（Data Exploration）

このラボは*オプション*です。高度な Pandas の使用法と詳細なデータ分析を実演します。

---

学習目標:

1. モデリング前にデータを探索するための有用なパターンを学ぶ
2. データセットへの理解を深め、データの問題点を特定する

このノートブックの目標は、特徴量エンジニアリングとモデリングを開始する前にベーステーブルを探索することです。S&P 500 に含まれる銘柄の株価履歴を探索します。

* 株価履歴（Price history）: 銘柄の株価履歴
* S&P 500 : S&P 500 に含まれるすべての企業と銘柄シンボルのリスト

分析のために、2000 年以降の株価履歴に限定しましょう。一般に、古い履歴データを使用するほど、その予測力は低下する可能性があります。

In [ ]:
import os

PROJECT = !(gcloud config get-value core/project)
PROJECT = PROJECT[0]
BUCKET = PROJECT

os.environ["PROJECT"] = PROJECT
os.environ["BUCKET"] = BUCKET

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from google.cloud import bigquery
from IPython import get_ipython
from IPython.core.magic import register_cell_magic
from matplotlib import pyplot as plt

%load_ext google.cloud.bigquery

bq = bigquery.Client(project=PROJECT)

In [ ]:
# SQL クエリ内で Python 変数を簡単に使用できるようにします。
@register_cell_magic("with_globals")
def with_globals(line, cell):
    contents = cell.format(**globals())
    if "print" in line:
        print(contents)
    get_ipython().run_cell(contents)

## データセットの準備

次のセルを実行して、プロジェクトの BigQuery にデータセットを作成し、株価データをインポートしましょう:

In [ ]:
!bq mk stock_src

In [ ]:
%%bash

TABLE=price_history
SCHEMA=symbol:STRING,Date:DATE,Open:FLOAT,Close:FLOAT

gcloud storage cp gs://asl-public-data/data/stock/$TABLE.csv gs://$BUCKET/stock_src/$TABLE.csv
bq load --source_format=CSV --skip_leading_rows=1 \
    stock_src.$TABLE gs://$BUCKET/stock_src/$TABLE.csv  $SCHEMA


In [ ]:
%%bash

TABLE=eps
SCHEMA=date:DATE,company:STRING,symbol:STRING,surprise:STRING,reported_EPS:FLOAT,consensus_EPS:FLOAT

gcloud storage cp gs://asl-public-data/data/stock/$TABLE.csv gs://$BUCKET/stock_src/$TABLE.csv
bq load --source_format=CSV --skip_leading_rows=1 \
    stock_src.$TABLE gs://$BUCKET/stock_src/$TABLE.csv  $SCHEMA


In [ ]:
%%bash

TABLE=snp500
SCHEMA=company:STRING,symbol:STRING,industry:STRING

gcloud storage cp gs://asl-public-data/data/stock/$TABLE.csv gs://$BUCKET/stock_src/$TABLE.csv
bq load --source_format=CSV --skip_leading_rows=1 \
    stock_src.$TABLE gs://$BUCKET/stock_src/$TABLE.csv  $SCHEMA


分析対象のテーブルと列を確認しましょう。`INFORMATION_SCHEMA` にクエリを実行してください。

**学習目標 1.**

In [ ]:
%%with_globals
%%bigquery --project {PROJECT}
SELECT table_name, column_name, data_type
FROM `stock_src.INFORMATION_SCHEMA.COLUMNS`
ORDER BY table_name, ordinal_position

## 株価履歴（Price History）

**TODO**: データセットの銘柄シンボルを可視化します。

In [ ]:
%%with_globals
%%bigquery --project {PROJECT}
SELECT *
FROM `stock_src.price_history`
LIMIT 10

In [ ]:
def query_stock(symbol):
    return bq.query(
        """
    # TODO: 特定の銘柄をクエリする
    """.format(
            symbol
        )
    ).to_dataframe()

In [ ]:
df_stock = query_stock("GOOG")

In [ ]:
df_stock.Date = pd.to_datetime(df_stock.Date)
ax = df_stock.plot(x="Date", y="Close", title="price")

# 平滑化プロットを追加。
df_stock["Close_smoothed"] = df_stock.Close.rolling(100, center=True).mean()
df_stock.plot(x="Date", y="Close_smoothed", ax=ax);

**TODO 2**: 個別銘柄を S&P 500 と比較します。

In [ ]:
SP500_SYMBOL = "gspc"
df_sp = query_stock(SP500_SYMBOL)

# TODO: S&P 500 の価格を可視化する

年単位で株価が時間の経過とともにどのように変化するかを見てみましょう。`LAG` 関数を使用すると、前年比（前年同期比）の株価変化を計算できます。

各年の平均終値の差を計算しましょう。この処理は当然 Pandas でも実行可能です。探索的データ分析では、BigQuery と Pandas を組み合わせて使用すると便利なことが多々あります。一般に、高負荷の処理は BigQuery に任せ、小規模なデータ処理や可視化には Pandas を使用するのが最も効果的です。

**学習目標 1, 2**

In [ ]:
%%with_globals
%%bigquery df --project {PROJECT}
WITH 
with_year AS
    (
    SELECT symbol, 
    EXTRACT(YEAR FROM date) AS year,
    close
    FROM `stock_src.price_history`
    WHERE symbol in (SELECT symbol FROM `stock_src.snp500`)
    ),
year_aggregated AS
    (
    SELECT year, symbol, AVG(close) as avg_close
    FROM with_year
    WHERE year >= 2000
    GROUP BY year, symbol
    )
SELECT year, symbol, avg_close as close,
(LAG( 
    --# TODO: avg_close の 1 年ラグを計算する
)) 
    AS next_yr_close
FROM year_aggregated
ORDER BY symbol, year

前年比の増加率（パーセンテージ）を計算します。

In [ ]:
df.dropna(inplace=True)
df["percent_increase"] = (df.next_yr_close - df.close) / df.close

年ごとの株価を可視化してみましょう

In [ ]:
def get_random_stocks(n=5):
    random_stocks = df.symbol.sample(n=n, random_state=3)
    rand = df.merge(random_stocks)
    return rand[["year", "symbol", "percent_increase"]]


rand = get_random_stocks()
for symbol, _df in rand.groupby("symbol"):
    plt.figure()
    sns.barplot(x="year", y="percent_increase", data=_df)
    plt.title(symbol)

個別銘柄にはいくつかの大きな変動がありました。たとえば、2000年代初頭にはハイテク企業の大幅な下落が見られました。

In [ ]:
df.sort_values("percent_increase").head()

In [ ]:
stock_symbol = "YHOO"

In [ ]:
%%with_globals
%%bigquery df --project {PROJECT}
SELECT date, close
FROM `stock_src.price_history`
WHERE symbol='{stock_symbol}'
ORDER BY date

In [ ]:
ax = df.plot(x="date", y="close")

**株式分割（Stock splits）**もデータに影響を与え、株価が急落する原因となります。実際には、これを考慮してすべての株価データをクリーンアップする必要があります。これは多大な労力を要します！幸いなことに、たとえば [IBM](https://www.fool.com/investing/2017/01/06/ibm-stock-split-will-2017-finally-be-the-year-shar.aspx) の場合、すべての株式分割は 2000 年以前に発生しています。

**学習目標 2**

**TODO**: IBM の株価履歴をクエリし、株式分割がデータにどのように影響するかを可視化してください。株式分割が発生すると、価格が急激に下落します。

In [ ]:
stock_symbol = "IBM"

In [ ]:
%%with_globals
%%bigquery df --project {PROJECT}
SELECT date, close
FROM `stock_src.price_history`
WHERE symbol='{stock_symbol}'
ORDER BY date

In [ ]:
# TODO: 主な株式分割がいつ発生したかを可視化できますか？

## S&P 採用銘柄企業リスト

In [ ]:
%%with_globals
%%bigquery df --project {PROJECT}
SELECT *
FROM `stock_src.snp500`

In [ ]:
df.industry.value_counts().plot(kind="barh");

株価履歴テーブルと S&P 500 テーブルを結合して、業種（industry）ごとに比較できます:

**学習目標 1, 2**

In [ ]:
%%with_globals
%%bigquery df --project {PROJECT}
WITH sp_prices AS
    (
    SELECT a.*, b.industry
    FROM `stock_src.price_history` a 
        JOIN `stock_src.snp500` b
    USING (symbol)
    WHERE date >= "2000-01-01"
    )
SELECT Date, industry, AVG(close) as close
FROM sp_prices
GROUP BY Date, industry
ORDER BY industry, Date

In [ ]:
df.head()

Pandas を使用してテーブルを「unstack」し、各業種を独自の列にすることができます。これはプロットに便利です。

In [ ]:
# 各業種を列にするための Pandas の `unstack`。プロットに便利です。
df_ind = df.set_index(["industry", "Date"]).unstack(0).dropna()
df_ind.columns = [c[1] for c in df_ind.columns]
df_ind.head()

In [ ]:
ax = df_ind.plot(figsize=(16, 8))
# 凡例を下に移動。
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.05), shadow=True, ncol=2)

最小最大スケーリング（Min-Max スケーリング）を使用して各業種をスケーリングしましょう。これにより、すべての銘柄が同じスケールに揃えられます。現状のままでは、業種間の経時的な株価変化を確認するのが難しい場合があります。

**学習目標 1**

In [ ]:
def min_max_scale(df):
    return (df - df.min()) / df.max()


scaled = min_max_scale(df_ind)
ax = scaled.plot(figsize=(16, 8))
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.05), shadow=True, ncol=2);

[移動平均（rolling mean）](https://en.wikipedia.org/wiki/Moving_average) を使用して、上記のプロットの平滑化バージョンを作成することもできます。これは、時系列データを可視化する際に役立つ変換です。

In [ ]:
SMOOTHING_WINDOW = 30  # 日数。
rolling = scaled.copy()
for col in scaled.columns:
    rolling[col] = scaled[col].rolling(SMOOTHING_WINDOW).mean()
ax = rolling.plot(figsize=(16, 8))
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.05), shadow=True, ncol=2);

情報技術（Information technology）は、他のすべての銘柄とともに、2000年代初頭および2008/2009年に大規模な暴落を経験しました。2008年以降、一部の業種は他の業種よりも回復がやや遅れました。

ボーナス: 次のラボでは、将来の株価を予測します。将来の価格を予測するためにどのような特徴量を使用できるでしょうか？これらの特徴量のいくつかを可視化してみてください。

Copyright 2019 Google Inc. Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0 Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License